In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from xgboost import XGBRegressor

# ۱. خواندن فایل اصلی
file_path = r'second_stage_inputs\G12\dsas_g12_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G12\dsas_g12_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g12_bearings_vibration_temp_deviation_monitoring_output2.xlsx'
try:
    df_raw = pd.read_excel(file_path)
    print("فایل اصلی با موفقیت خوانده شد.")
except Exception as e:
    print(f"خطا: {e}")
    exit()

# لیست فیچرها و تارگت‌ها
all_features = [
     'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
    'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
    'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
]
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# ۲. حذف نویز و داده‌های پرت (DBSCAN)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {i: scaled_data[labels == i].mean(axis=0) for i in set(labels) if i != -1}
def calc_dist(idx):
    label = labels[idx]
    point = scaled_data[idx].reshape(1, -1)
    if label != -1:
        return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0

df_raw['distance'] = [calc_dist(i) for i in range(len(df_raw))]
df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
df_cleaned = df_cleaned.sort_values(by='date')

# ۳. آموزش مدل و پیش‌بینی
split_idx = int(len(df_cleaned) * 0.80)
train_df = df_cleaned.iloc[:split_idx].copy()
test_df = df_cleaned.iloc[split_idx:].copy()

for target in target_sensors:
    X_train, y_train = train_df[[f for f in all_features if f != target]], train_df[target]
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    # ایجاد ستون پیش‌بینی با نام موقت برای Unpivot کردن راحت‌تر
    test_df[f'PRED_{target}'] = model.predict(test_df[[f for f in all_features if f != target]])

# ۴. تغییر ساختار داده (Unpivot/Melt) به فرمت درخواستی شما
# مرحله اول: استخراج مقادیر واقعی (Actual)
melted_actual = test_df.melt(id_vars=['date'], value_vars=target_sensors, 
                             var_name='AssetID', value_name='actual')

# مرحله دوم: استخراج مقادیر پیش‌بینی شده (Predicted)
pred_cols = [f'PRED_{t}' for t in target_sensors]
melted_pred = test_df.melt(id_vars=['date'], value_vars=pred_cols, 
                           var_name='temp_target', value_name='predicted')


# اصلاح نام AssetID در دیتافریم پیش‌بینی برای Merge کردن (حذف PRED_)
melted_pred['AssetID'] = melted_pred['temp_target'].str.replace('PRED_', '')

# مرحله سوم: ادغام دو دیتافریم بر اساس تاریخ و نام سنسور
final_long_df = pd.merge(melted_actual, melted_pred[['date', 'AssetID', 'predicted']], 
                         on=['date', 'AssetID'])

# ۵. ذخیره خروجی نهایی
final_long_df.to_excel(output_filename, index=False)

print("فایل جدید با ساختار مورد نظر شما (date, AssetID, actual, predicted) ایجاد شد.")


فایل اصلی با موفقیت خوانده شد.
فایل جدید با ساختار مورد نظر شما (date, AssetID, actual, predicted) ایجاد شد.
